In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1) Download DINO-WM deformable dataset (HF mirror, byte-identical to OSF) to local disk
!df -h /content | tail -1
!pip -q install -U huggingface_hub hf_transfer
import os; os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
from huggingface_hub import hf_hub_download
files = ["deformable.z01", "deformable.z02", "deformable.z03", "deformable.zip"]
paths = [hf_hub_download("gaoyuezhou/dino_wm", f"datasets/{f}", repo_type="dataset", local_dir="/content/dwm") for f in files]
print(paths)

In [ ]:
# 2) Join the split archive, list its contents, extract ONE rope episode
%cd /content/dwm/datasets
!apt-get -qq install -y zip > /dev/null
!zip -s- deformable.zip -O deformable_full.zip && rm deformable.z0* deformable.zip
!unzip -l deformable_full.zip | head -25
!unzip -l deformable_full.zip | grep "rope/000000/"
!unzip -q -o deformable_full.zip "*rope/000000/*" -d /content/rope_ep0

In [ ]:
# 6) Pictures: RGB + depth (camera 1) for rope and granular. Depth in meters = uint16 / 10000 (1 sim unit = 0.1 m, robot URDF loaded with globalScaling=10)
import h5py, glob
!unzip -q -o /content/dwm/datasets/deformable_full.zip "*granular/000000/*" -d /content/gran_ep0
import matplotlib.pyplot as plt
def load(ep, t, cam="cam_1"): f = h5py.File(f"{ep}/{t:02d}.h5", "r"); return f[f"observations/color/{cam}"][0] / 255, f[f"observations/depth/{cam}"][0] / 10000
eps = {"rope": glob.glob("/content/rope_ep0/**/rope/000000", recursive=True)[0], "granular": glob.glob("/content/gran_ep0/**/granular/000000", recursive=True)[0]}
frames = [0, 5, 10, 19]
fig, ax = plt.subplots(4, 4, figsize=(16, 16))
for i, (name, t) in enumerate([(n, t) for n in eps for t in frames]): rgb, dep = load(eps[name], t); r, c = (i // 4) * 2, i % 4; ax[r, c].imshow(rgb); ax[r, c].set_title(f"{name} t={t} RGB"); im = ax[r + 1, c].imshow(dep, cmap="viridis"); ax[r + 1, c].set_title(f"depth {dep.min():.2f}-{dep.max():.2f} m"); fig.colorbar(im, ax=ax[r + 1, c], fraction=0.046)
[a.axis("off") for a in ax.flat]; plt.tight_layout(); plt.show()
fig2, ax2 = plt.subplots(2, 4, figsize=(16, 8))
for j in range(4): rgb, dep = load(eps["rope"], 10, f"cam_{j}"); ax2[0, j].imshow(rgb); ax2[0, j].set_title(f"rope t=10 cam_{j}"); im = ax2[1, j].imshow(dep, cmap="viridis"); ax2[1, j].set_title(f"depth {dep.min():.2f}-{dep.max():.2f} m"); fig2.colorbar(im, ax=ax2[1, j], fraction=0.046)
[a.axis("off") for a in ax2.flat]; plt.tight_layout(); plt.show()
pos = h5py.File(f"{eps['rope']}/00.h5", "r")["positions"][0][:, :3]; print("rope particle extent x,y,z (sim units):", pos.max(0) - pos.min(0), "-> expect rope_length 2.5 sim = 0.25 m")

In [ ]:
# 12) What DeFM actually receives: its 3 input channels for one Granular frame (camera 1),
#     computed by DeFM's own convert_to_3channel_metric (before its final mean/std normalization)
#     Needs cells 7-8 (DeFM import + extracted Granular data); queued to run after the pilot.
from defm import convert_to_3channel_metricxpla
import matplotlib.pyplot as plt
ep, t = "/content/data/deformable/granular/000000", 10
with h5py.File(f"{ep}/{t:02d}.h5", "r") as f:
    rgb_img = f["observations/color/cam_1"][0] / 255
    depth_m = f["observations/depth/cam_1"][0] / 10000.0          # meters (1 sim unit = 0.1 m)
ch = convert_to_3channel_metric(depth_m)                           # (224, 224, 3): C1, C2, C3
names = ["C1 = log(1+d) / log(1+100 m)", "C2 = log(1+d) / log(1+9 m), clipped to 1", "C3 = per-image min-max of log(1+d)"]
fig, ax = plt.subplots(2, 5, figsize=(24, 9.5))
ax[0, 0].imshow(rgb_img); ax[0, 0].set_title(f"RGB, granular episode 0, t={t}")
im = ax[0, 1].imshow(depth_m, cmap="viridis"); ax[0, 1].set_title(f"raw depth {depth_m.min():.2f}-{depth_m.max():.2f} m"); fig.colorbar(im, ax=ax[0, 1], fraction=0.046)
for k in range(3):
    im = ax[0, k + 2].imshow(ch[..., k], cmap="gray", vmin=0, vmax=1)
    ax[0, k + 2].set_title(names[k] + "\nshown on the full 0-1 scale", fontsize=10); fig.colorbar(im, ax=ax[0, k + 2], fraction=0.046)
    im = ax[1, k + 2].imshow(ch[..., k], cmap="gray")
    ax[1, k + 2].set_title(f"C{k + 1} contrast-stretched to its own range\n{ch[..., k].min():.3f} to {ch[..., k].max():.3f}", fontsize=10); fig.colorbar(im, ax=ax[1, k + 2], fraction=0.046)
for k in range(3):
    ax[1, 0].hist(ch[..., k].ravel(), bins=100, range=(0, 1), alpha=0.6, label=f"C{k + 1}")
ax[1, 0].set(title="Which part of 0-1 each channel uses", xlabel="channel value", ylabel="pixels"); ax[1, 0].legend()
ax[1, 1].axis("off")
for a in list(ax[0]) + list(ax[1, 2:]): a.axis("off")
plt.tight_layout()
out_dir = globals().get("out", "/content/depth_pilot"); os.makedirs(out_dir, exist_ok=True)
plt.savefig(f"{out_dir}/defm_channels.png", dpi=110); plt.show()
for k in range(3):
    lo, hi = ch[..., k].min(), ch[..., k].max()
    print(f"C{k + 1}: {lo:.3f} to {hi:.3f}  -> uses {100 * (hi - lo):.0f}% of the 0-1 range")

In [ ]:
# 7) Depth pilot setup: GPU check, code (DINO-WM fork + DeFM), imports
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip -q install omegaconf
!test -d /content/dino_wm || git clone -q https://github.com/IvanFenster/dino_wm.git /content/dino_wm
!test -d /content/defm || git clone -q https://github.com/leggedrobotics/defm.git /content/defm
import os, sys, glob, json, time
sys.path.insert(0, "/content/dino_wm"); sys.path.insert(0, "/content/defm")
import numpy as np, h5py, torch, torch.nn.functional as F
import torchvision.transforms as T
from models.vit import ViTPredictor                    # DINO-WM predictor (unchanged)
from models.proprio import ProprioceptiveEmbedding     # DINO-WM action / proprio encoders
from defm import preprocess_depth_image                # DeFM 3-channel metric depth format
from defm.model_factory import create_defm_model
dev = "cuda"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 8) Extract N_EP Granular episodes + dataset-level actions/states (needs the joined zip from cell 2)
N_EP = 200
ZIP = "/content/dwm/datasets/deformable_full.zip"
pats = " ".join(f'"deformable/granular/{i:06d}/*"' for i in range(N_EP))
!unzip -q -o {ZIP} "deformable/granular/actions.pth" "deformable/granular/states.pth" {pats} -x "*/20.h5" -d /content/data
!du -sh /content/data/deformable/granular; ls /content/data/deformable/granular | head -5

In [ ]:
# 9) Encode every frame ONCE with the frozen encoders (pipeline steps 1-3), cache tokens on the GPU
root = "/content/data/deformable/granular"
actions = torch.load(f"{root}/actions.pth").float()[:N_EP]                        # (N, T, 4)
print("actions", tuple(actions.shape))
dino = torch.hub.load("facebookresearch/dinov2", "dinov2_vits14").to(dev).eval().half()
defm = create_defm_model("defm_vit_s14", pretrained=True).to(dev).eval().half()
rgb_tf = T.Compose([T.Resize(224), T.CenterCrop(224), T.Normalize([0.5] * 3, [0.5] * 3)])  # DINO-WM data loader
to196 = T.Resize(196, antialias=True)                                                    # DINO-WM model resize, widened to DeFM

@torch.no_grad()
def enc(model, x):
    return model.forward_features(x.half())["x_norm_patchtokens"]                       # (B, 196, 384)

feats_rgb, feats_dep, t0 = [], [], time.time()
for i in range(N_EP):
    ep = f"{root}/{i:06d}"
    rgb = torch.load(f"{ep}/obses.pth").permute(0, 3, 1, 2) / 255.0                   # (20, 3, 224, 224)
    rgb = to196(rgb_tf(rgb)).to(dev)
    dep_m = [h5py.File(f"{ep}/{t:02d}.h5", "r")["observations/depth/cam_1"][0] / 10000.0 for t in range(20)]  # meters
    dep = torch.cat([preprocess_depth_image(d, target_size=224) for d in dep_m])       # (20, 3, 224, 224) DeFM format
    dep = to196(dep).to(dev)
    feats_rgb.append(enc(dino, rgb)); feats_dep.append(enc(defm, dep))
    if i == 0: print("per frame:", tuple(rgb.shape[1:]), "->", tuple(feats_rgb[0].shape[1:]), "| depth", tuple(dep.shape[1:]), "->", tuple(feats_dep[0].shape[1:]))
feats_rgb, feats_dep = torch.stack(feats_rgb), torch.stack(feats_dep)                  # (N, 20, 196, 384)
print("RGB tokens", tuple(feats_rgb.shape), "| depth tokens", tuple(feats_dep.shape), f"| {time.time() - t0:.0f}s")
s_rgb, s_dep = feats_rgb.float().std().item(), feats_dep.float().std().item()
print(f"feature std: DINOv2 {s_rgb:.3f} | DeFM {s_dep:.3f} | ratio {s_dep / s_rgb:.2f}")

In [ ]:
# 10) Train DINO-WM's predictor on the cached tokens. H=1, frameskip=1 (paper Table 11), concat_dim=1,
#     lrs 5e-5 / 5e-4, batch 32. Variants: RGB only (baseline), depth only, RGB+depth concat.
a_flat = actions.reshape(-1, 4)
acts = ((actions - a_flat.mean(0)) / (a_flat.std(0) + 1e-6)).to(dev)            # normalize_action=True
n_tr = int(0.9 * N_EP)                                                            # split_ratio 0.9
scale_dep = s_rgb / s_dep                     # fixed constant: puts the DeFM half on DINOv2's scale (nothing trained)
T_ACT = acts.shape[1]

def visual(variant, ep, t):
    if variant == "rgb": return feats_rgb[ep, t].float()
    if variant == "depth": return feats_dep[ep, t].float()
    return torch.cat([feats_rgb[ep, t].float(), feats_dep[ep, t].float() * scale_dep], -1)  # (B, 196, 768)

def make_pairs(eps): return torch.tensor([(e, t) for e in eps for t in range(19)])
tr_pairs, va_pairs = make_pairs(range(n_tr)), make_pairs(range(n_tr, N_EP))

def run(variant, epochs=5, bs=32):
    D = 768 if variant == "concat" else 384
    torch.manual_seed(0)
    pred = ViTPredictor(num_patches=196, num_frames=1, dim=D + 20, depth=6, heads=16, mlp_dim=2048,
                        dropout=0.1, emb_dropout=0, pool="mean").to(dev)
    act_enc = ProprioceptiveEmbedding(num_frames=1, tubelet_size=1, in_chans=4, emb_dim=10).to(dev)
    prop_enc = ProprioceptiveEmbedding(num_frames=1, tubelet_size=1, in_chans=1, emb_dim=10).to(dev)
    opt_p = torch.optim.AdamW(pred.parameters(), lr=5e-5)
    opt_a = torch.optim.AdamW(list(act_enc.parameters()) + list(prop_enc.parameters()), lr=5e-4)
    scaler = torch.amp.GradScaler("cuda")

    def z_of(ep, t):  # DINO-WM token = [visual | proprio emb | action emb], copied onto all 196 patches
        v = visual(variant, ep, t)
        p = prop_enc(torch.zeros(len(ep), 1, 1, device=dev)).expand(-1, 196, -1)                # dummy proprio
        a = act_enc(acts[ep, t.clamp(max=T_ACT - 1)].unsqueeze(1)).expand(-1, 196, -1)
        return torch.cat([v, p, a], -1)

    def losses(ep, t):
        z_in, z_tgt = z_of(ep, t), z_of(ep, t + 1).detach()
        z_hat = pred(z_in)
        out = {"loss": F.mse_loss(z_hat[..., :-10], z_tgt[..., :-10]),        # DINO-WM: all but the action dims
               "visual": F.mse_loss(z_hat[..., :D], z_tgt[..., :D]),
               "copy_baseline": F.mse_loss(z_in[..., :D], z_tgt[..., :D])}  # guess "next frame = this frame"
        if variant == "concat":
            out["rgb_half"] = F.mse_loss(z_hat[..., :384], z_tgt[..., :384])
            out["depth_half"] = F.mse_loss(z_hat[..., 384:768], z_tgt[..., 384:768])
        return out

    log = {"variant": variant, "D": D, "params_M": sum(p.numel() for p in pred.parameters()) / 1e6, "train": [], "val": []}
    step, t0 = 0, time.time()
    for epoch in range(epochs):
        pred.train()
        for b in tr_pairs[torch.randperm(len(tr_pairs))].split(bs):
            ep, t = b[:, 0].to(dev), b[:, 1].to(dev)
            with torch.autocast("cuda", dtype=torch.float16):
                L = losses(ep, t)
            opt_p.zero_grad(); opt_a.zero_grad()
            scaler.scale(L["loss"]).backward(); scaler.step(opt_p); scaler.step(opt_a); scaler.update()
            step += 1
            if step % 10 == 0: log["train"].append({"step": step, **{k: v.item() for k, v in L.items()}})
        pred.eval(); agg = {}
        with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16):
            for b in va_pairs.split(64):
                for k, v in losses(b[:, 0].to(dev), b[:, 1].to(dev)).items():
                    agg[k] = agg.get(k, 0.0) + v.item() * len(b)
        val = {k: v / len(va_pairs) for k, v in agg.items()}
        val["relative"] = val["visual"] / val["copy_baseline"]
        log["val"].append({"epoch": epoch + 1, "step": step, **val})
        extra = f" | rgb_half {val['rgb_half']:.4f} depth_half {val['depth_half']:.4f}" if variant == "concat" else ""
        print(f"[{variant}] epoch {epoch + 1} step {step} | val visual {val['visual']:.4f} | copy {val['copy_baseline']:.4f} | relative {val['relative']:.3f}{extra} | {time.time() - t0:.0f}s")
    log["time_s"] = time.time() - t0
    return log

print(f"train pairs {len(tr_pairs)}, val pairs {len(va_pairs)}")
logs = {v: run(v) for v in ["rgb", "depth", "concat"]}

In [ ]:
# 11) Results: curves + summary table; save logs and plot (Google Drive if mounted, else /content)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
for v, lg in logs.items():
    ax[0].plot([r["step"] for r in lg["train"]], [r["visual"] for r in lg["train"]], label=f"{v} (D={lg['D']})")
    ax[1].plot([r["epoch"] for r in lg["val"]], [r["relative"] for r in lg["val"]], marker="o", label=v)
ax[0].set(title="Train loss on visual tokens (scales differ across variants)", xlabel="step", ylabel="MSE", yscale="log")
ax[1].set(title="Val loss / 'next = current' loss (comparable; lower is better)", xlabel="epoch", ylabel="relative loss")
ax[1].axhline(1.0, ls="--", c="gray"); [a.legend() for a in ax]; plt.tight_layout()
out = "/content/drive/MyDrive/WM_research/depth_pilot" if os.path.isdir("/content/drive/MyDrive") else "/content/depth_pilot"
os.makedirs(out, exist_ok=True); plt.savefig(f"{out}/curves.png", dpi=120); plt.show()
print(f"{'variant':8} {'token D':>7} {'pred M':>7} {'val MSE':>9} {'copy MSE':>9} {'relative':>9} {'time s':>7}")
for v, lg in logs.items():
    f = lg["val"][-1]
    print(f"{v:8} {lg['D']:7d} {lg['params_M']:7.1f} {f['visual']:9.4f} {f['copy_baseline']:9.4f} {f['relative']:9.3f} {lg['time_s']:7.0f}")
c, r = logs["concat"]["val"][-1], logs["rgb"]["val"][-1]
print(f"\nSame-space check (both in DINOv2 space): RGB-only val {r['visual']:.4f} vs RGB half of concat {c['rgb_half']:.4f}")
json.dump({"N_EP": N_EP, "s_rgb": s_rgb, "s_dep": s_dep, "scale_dep": scale_dep, "logs": logs}, open(f"{out}/logs.json", "w"), indent=1)
print("saved to", out)